# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | khaleelalaqel| 
| **Mehmet Kilinc 25135007** | ** |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [7]:
# Gedeelde imports (door iedereen te gebruiken/aan te vullen)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (6, 4)

RANDOM_STATE = 42 


**pd.set_option** zorgt ervoor dat pandas alle kolommen toont wanneer je een dataframe print, in plaats van kolommen af te korten met ...
<br>

**plt.rcParams** stelt de standaardgrootte in van elke grafiek die we met matplotlib maken

<br>

### Data inladen

In [8]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




------------

### 2.4 Evaluatiefuncties
Deze functies gebruiken we bij alle modellen:

- `get_scores()` geeft per patiënt een score: de kans op stroke (`predict_proba`) of, bij een SVM, de afstand tot het hypervlak (`decision_function`).
- `find_best_threshold()` zoekt de **beslisdrempel** met de hoogste F1-score. Een model voorspelt `stroke = 1` als de score boven die drempel ligt. Waarom dit belangrijk is, staat in paragraaf 5.1.
- `F1_BEST_SCORER` is de score die `GridSearchCV` gebruikt: de F1-score bij de beste drempel.
- `metrics_from_confusion()` rekent de metrics **zelf** uit met de formules uit paragraaf 4.2, zodat we kunnen laten zien hoe ze worden berekend.

In [ ]:
def get_scores(model, features):
    """Geeft per patiënt een score: kans op stroke of decision function."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    return model.decision_function(features)


def f1_per_threshold(y_true, scores):
    """Berekent de F1-score voor iedere mogelijke beslisdrempel."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    # Het laatste punt van de curve hoort niet bij een drempel, want P=1 , R=0 en geen threshold. 
    f1_values = (2 * precision[:-1] * recall[:-1]
                 / np.maximum(precision[:-1] + recall[:-1], 1e-12))
    return thresholds, f1_values


def find_best_threshold(y_true, scores):
    """Geeft de drempel met de hoogste F1-score en die F1-score."""
    thresholds, f1_values = f1_per_threshold(y_true, scores)
    best = np.argmax(f1_values)
    return thresholds[best], f1_values[best]


def f1_at_best_threshold(y_true, scores):
    """Scorefunctie voor GridSearchCV: F1 bij de beste drempel."""
    return find_best_threshold(y_true, scores)[1]


F1_BEST_SCORER = make_scorer(
    f1_at_best_threshold,
    response_method=("predict_proba", "decision_function"))


def metrics_from_confusion(tn, fp, fn, tp):
    """Berekent de metrics zelf uit de vier cellen van de matrix."""
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    specificity = tn / (tn + fp) if tn + fp > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0.0)
    return {"accuracy": accuracy, "precision": precision,
            "recall": recall, "specificity": specificity, "f1": f1}


def evaluate_predictions(y_true, y_pred, scores):
    """Combineert de metrics uit de confusion matrix met ROC- en PR-AUC."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    results = metrics_from_confusion(tn, fp, fn, tp)
    results["roc_auc"] = roc_auc_score(y_true, scores)
    results["pr_auc"] = average_precision_score(y_true, scores)
    results.update({"TP": tp, "FP": fp, "FN": fn, "TN": tn})
    return results

### 2.5 Klassen `ModelExperiment` en `ResultsTable`
Ieder model wordt op **precies dezelfde manier** getraind, getest en beoordeeld. Daarvoor gebruiken we de klasse `ModelExperiment`. Een model doorloopt vier vaste stappen: hyperparameters tunen met 5-voudige gestratificeerde cross-validation, de beslisdrempel kiezen, testen op de validatieset (die tijdens het tunen niet is gebruikt) en een Kaggle-bestand maken. Doordat elk model dezelfde stappen volgt, kunnen we de resultaten eerlijk vergelijken. De klasse `ResultsTable` verzamelt de resultaten van alle modellen voor de eindtabel in hoofdstuk 6.

| Methode | Wat doet het? |
|---|---|
| `tune()` | **Stap 1:** zoekt de beste hyperparameters met `GridSearchCV` en 5-voudige gestratificeerde cross-validation op de trainingsset. |
| `tune_threshold()` | **Stap 2:** kiest met *out-of-fold* voorspellingen de beslisdrempel met de hoogste F1. |
| `evaluate()` | **Stap 3:** test het getunede model op de validatieset. |
| `make_submission()` | **Stap 4:** traint het model opnieuw op **alle** trainingsdata en maakt een csv-bestand voor Kaggle. |
| `run()` | Voert stap 1 t/m 4 achter elkaar uit en print de resultaten. |
| `best_params_text()` | Geeft de beste hyperparameters als korte, leesbare tekst (zonder `model__` ervoor). |
| `plot_evaluation()` | Toont de confusion matrix op de validatieset. |
| `summary()` | Geeft de resultaten als één rij voor de eindtabel. |
| `ResultsTable.add()` | Voegt een uitgevoerd model toe aan de tabel. |
| `ResultsTable.to_frame()` | Maakt de eindtabel met één rij per model. |


In [ ]:
class ModelExperiment:
    """Doorloopt voor één model de vier vaste stappen.

    name: naam van het model, pipeline: scaler/sampler + model,
    param_grid: hyperparameters voor GridSearchCV,
    techniques: technieken tegen de imbalance (tekst voor de eindtabel).
    """

    def __init__(self, name, pipeline, param_grid, techniques):
        self.name = name
        self.pipeline = pipeline
        self.param_grid = param_grid
        self.techniques = techniques
        self.cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                  random_state=RANDOM_STATE)

    def tune(self, X, y):
        """Stap 1: beste hyperparameters zoeken met GridSearchCV."""
        start = time.perf_counter()
        self.search = GridSearchCV(self.pipeline, self.param_grid,
                                   scoring=F1_BEST_SCORER, cv=self.cv,
                                   n_jobs=N_JOBS)
        self.search.fit(X, y)
        self.tuning_seconds = time.perf_counter() - start
        self.best_model = self.search.best_estimator_

    def tune_threshold(self, X, y):
        """Stap 2: drempel kiezen met de hoogste out-of-fold F1."""
        if hasattr(self.best_model, "predict_proba"):
            method = "predict_proba"
        else:
            method = "decision_function"
        scores = cross_val_predict(clone(self.best_model), X, y,
                                   cv=self.cv, method=method, n_jobs=N_JOBS)
        if method == "predict_proba":
            scores = scores[:, 1]  # alleen de kans op stroke
        self.threshold, self.oof_f1 = find_best_threshold(y, scores)

    def evaluate(self, X, y):
        """Stap 3: het model testen op de validatieset."""
        scores = get_scores(self.best_model, X)
        self.val_pred = (scores >= self.threshold).astype(int)
        self.val_metrics = evaluate_predictions(y, self.val_pred, scores)
        # Ter vergelijking: F1 met de standaarddrempel (0,5 of 0)
        default = 0.5 if hasattr(self.best_model, "predict_proba") else 0.0
        self.val_metrics["f1_default"] = f1_score(
            y, (scores >= default).astype(int))

    def make_submission(self, X, y, X_test, test_ids):
        """Stap 4: opnieuw trainen op alle data en een csv maken."""
        final_model = clone(self.best_model).fit(X, y)
        scores = get_scores(final_model, X_test)
        predictions = (scores >= self.threshold).astype(int)
        file_name = "submission_" + self.name.lower().replace(" ", "_") + ".csv"
        self.submission_path = SUBMISSION_DIR / file_name
        pd.DataFrame({"id": test_ids, "stroke": predictions}).to_csv(
            self.submission_path, index=False)

    def run(self, data):
        """Voert stap 1 t/m 4 uit en print de resultaten."""
        self.tune(data["X_train"], data["y_train"])
        self.tune_threshold(data["X_train"], data["y_train"])
        self.evaluate(data["X_val"], data["y_val"])
        self.make_submission(data["X_full"], data["y_full"],
                             data["X_test"], data["test_ids"])
        m = self.val_metrics
        print(f"Model: {self.name}")
        print(f"Beste hyperparameters: {self.best_params_text()}")
        print(f"CV-F1: {self.search.best_score_:.3f}")
        print(f"Gekozen drempel: {self.threshold:.3f}")
        print(f"Validatie-F1: {m['f1']:.3f} met gekozen drempel, "
              f"{m['f1_default']:.3f} met standaarddrempel")
        print(f"Precision: {m['precision']:.3f}, recall: {m['recall']:.3f}")
        print(f"Tuningtijd: {self.tuning_seconds:.0f} s | "
              f"submission: {self.submission_path}")
        return self

    def best_params_text(self):
        """Beste hyperparameters als tekst, zonder 'model__' ervoor."""
        return ", ".join(f"{key.split('__')[-1]}={value}"
                         for key, value in self.search.best_params_.items())

    def plot_evaluation(self, y_val):
        """Toont de confusion matrix op de validatieset."""
        ConfusionMatrixDisplay.from_predictions(
            y_val, self.val_pred, display_labels=["Geen stroke", "Stroke"],
            cmap="Blues", colorbar=False)
        plt.title(f"{self.name}: validatieset (drempel = {self.threshold:.3f})")
        plt.xlabel("Voorspeld")
        plt.ylabel("Werkelijk")
        plt.grid(False)
        plt.show()

    def summary(self):
        """Resultaten als één rij voor de eindtabel."""
        m = self.val_metrics
        return {"model": self.name,
                "technieken imbalance": self.techniques,
                "CV-F1": self.search.best_score_,
                "validatie-F1": m["f1"],
                "precision": m["precision"],
                "recall": m["recall"],
                "ROC-AUC": m["roc_auc"],
                "PR-AUC": m["pr_auc"],
                "drempel": self.threshold,
                "beste hyperparameters": self.best_params_text()}


class ResultsTable:
    """Verzamelt de resultaten van alle modellen in één tabel."""

    def __init__(self):
        self.experiments = {}

    def add(self, experiment):
        """Voegt een model toe (opnieuw draaien overschrijft de oude rij)."""
        self.experiments[experiment.name] = experiment

    def to_frame(self, kaggle_scores=None):
        """Maakt een DataFrame met één rij per model."""
        rows = [exp.summary() for exp in self.experiments.values()]
        table = pd.DataFrame(rows).set_index("model")
        if kaggle_scores is not None:
            table.insert(2, "Kaggle-F1", pd.Series(kaggle_scores))
        return table

## 4. Opdracht 2 – Evaluatie

### 4.1 Train- en validatieset
We splitsen `train.csv` in een **trainingsset (80%)** en een **validatieset (20%)**. De split is **gestratificeerd**: in beide delen zit hetzelfde percentage stroke (1,54%). Zonder stratificatie kan de kleine groep stroke-patiënten toevallig scheef verdeeld raken.

- Met de **trainingsset** zoeken we hyperparameters en de beslisdrempel (met cross-validation).
- De **validatieset** gebruiken we pas aan het einde, als eerlijke test op data die het model nooit heeft gezien. Deze score noemen we in de eindtabel de *notebookscore*.
- Voor Kaggle trainen we het definitieve model op **alle** 33.544 patiënten.

In [1]:

X_train, X_val, y_train, y_val = dataset.split(test_size=0.2)
X_full, y_full = dataset.get_features_and_target()

# Alle splits in één dict, zodat ieder experiment dezelfde data krijgt
data = {"X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_full": X_full, "y_full": y_full,
        "X_test": dataset.test, "test_ids": dataset.test_ids}

split_overview = pd.DataFrame({
    "aantal patiënten": [len(y_train), len(y_val)],
    "aantal stroke": [y_train.sum(), y_val.sum()],
    "stroke (%)": [y_train.mean() * 100, y_val.mean() * 100]},
    index=["trainingsset", "validatieset"])
split_overview.round(2)

NameError: name 'dataset' is not defined

### 4.2 Metrics voor binaire classificatie

In deze casus is **positief (1) = stroke** en **negatief (0) = geen stroke**. Slechts ongeveer 1,5% van de patiënten heeft een stroke. Daarom zegt één getal zoals accuracy weinig; we kijken naar meerdere metrics.

#### Confusion matrix

|                        | **Voorspeld 0 (geen stroke)** | **Voorspeld 1 (stroke)** |
|------------------------|-------------------------------|--------------------------|
| **Werkelijk 0 (geen stroke)** | **TN** (True Negative): gezonde patiënt terecht gerustgesteld | **FP** (False Positive): vals alarm, gezonde patiënt onnodig doorverwezen |
| **Werkelijk 1 (stroke)**      | **FN** (False Negative): gemiste patiënt, stroke niet opgemerkt | **TP** (True Positive): stroke-patiënt terecht opgespoord |

In deze casus is een **FN** het ergst: een gemiste patiënt kan ernstige gevolgen hebben. Een **FP** kost vooral extra onderzoek en onrust.

#### Overzicht van de metrics

| Metric | Formule | Betekenis in deze casus |
|--------|---------|-------------------------|
| **Accuracy** | $\dfrac{TP + TN}{TP + TN + FP + FN}$ | Het deel van alle patiënten dat het model goed indeelt. Misleidend hier: een model dat iedereen "geen stroke" geeft, haalt al ongeveer 98,5%. |
| **Precision** | $\dfrac{TP}{TP + FP}$ | Van alle patiënten die het model als risico aanwijst, het deel dat echt een stroke krijgt. Lage precision betekent veel vals alarm. |
| **Recall (sensitiviteit)** | $\dfrac{TP}{TP + FN}$ | Van alle patiënten met een stroke, het deel dat het model vindt. Lage recall betekent veel gemiste patiënten. |
| **Specificiteit** | $\dfrac{TN}{TN + FP}$ | Van alle patiënten zonder stroke, het deel dat het model terecht geruststelt. Door de vele gezonde patiënten is deze bijna altijd hoog. |
| **F1-score** | $2 \cdot \dfrac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ | Eén getal dat precision en recall combineert. Is alleen hoog als het model zowel weinig patiënten mist als weinig vals alarm geeft. |
| **ROC-AUC** | $\displaystyle\int_0^1 \text{TPR} \; d(\text{FPR})$ | De kans dat het model een willekeurige stroke-patiënt een hoger risico geeft dan een willekeurige gezonde patiënt. 0,5 is gokken, 1 is perfect. |
| **PR-AUC** | $\displaystyle\int_0^1 \text{Precision} \; d(\text{Recall})$ | Laat zien hoe betrouwbaar de waarschuwingen blijven als we meer stroke-patiënten willen vinden. Gokken geeft hier ongeveer 0,015 (het aandeel stroke), dus elke stijging daarboven is echte winst. |

*Hierbij is TPR = recall en FPR = $\dfrac{FP}{FP + TN}$ = 1 − specificiteit.*

Accuracy, precision, recall, specificiteit en F1 hangen af van de gekozen beslisdrempel (bijvoorbeeld 0,5), terwijl ROC-AUC en PR-AUC het model over alle drempels heen beoordelen. Bij sterk imbalanced data is PR-AUC informatiever dan ROC-AUC, omdat ROC-AUC door het grote aantal terecht negatieven (TN) optimistisch hoog kan uitvallen, terwijl PR-AUC zich richt op de zeldzame positieve klasse en vals alarm direct zichtbaar maakt (Saito & Rehmsmeier, 2015, *PLoS ONE*).

(Claude, 2026. Prompt 1: &lt; https://claude.ai/share/f7ba8829-bae3-4e66-8cb7-176387434a36 &gt;)

### 4.3 Demonstratie met een zelfgekozen algoritme
We trainen een eenvoudige **logistic regression zonder technieken tegen de imbalance** en met de standaarddrempel 0,5. Daarna rekenen we de metrics zelf uit met `metrics_from_confusion()` en controleren we ze met de functies van scikit-learn.

In [ ]:
baseline = Pipeline([("scaler", StandardScaler()),
                     ("model", LogisticRegression(max_iter=1000))])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)          # drempel 0,5
baseline_scores = get_scores(baseline, X_val)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_val, baseline_pred, display_labels=["Geen stroke", "Stroke"],
    cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Baseline: logistic regression, drempel 0,5")
ax.set(xlabel="Voorspeld", ylabel="Werkelijk")
ax.grid(False)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = confusion_matrix(y_val, baseline_pred).ravel()
own = metrics_from_confusion(tn, fp, fn, tp)
sklearn_values = {
    "accuracy": accuracy_score(y_val, baseline_pred),
    "precision": precision_score(y_val, baseline_pred, zero_division=0),
    "recall": recall_score(y_val, baseline_pred),
    "specificity": recall_score(y_val, baseline_pred, pos_label=0),
    "f1": f1_score(y_val, baseline_pred)}
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"ROC-AUC = {roc_auc_score(y_val, baseline_scores):.3f}, "
      f"PR-AUC = {average_precision_score(y_val, baseline_scores):.3f}")
pd.DataFrame({"zelf berekend": own, "scikit-learn": sklearn_values}).round(4)

**Interpretatie.** Het baseline-model voorspelt voor **geen enkele** patiënt stroke: alle 103 stroke-patiënten in de validatieset worden gemist (FN = 103, TP = 0). Toch is de accuracy 98,5% en de specificiteit 100%. Precision, recall en F1 zijn 0. De zelf berekende metrics zijn gelijk aan die van scikit-learn.

De ROC-AUC van 0,83 laat zien dat het model patiënten wel redelijk kan **rangschikken** op risico. Het probleem is de drempel van 0,5: bij zo weinig stroke-patiënten krijgt niemand zo'n hoge kans. Dit laat zien waarom accuracy hier misleidend is en waarom we technieken tegen de imbalance nodig hebben (paragraaf 5.1).

### 4.4 Waarom is de F1-score de beste maatstaf voor dit vraagstuk?
1. **Accuracy is misleidend bij imbalance.** Zoals 4.3 laat zien, scoort een model dat niemand als risico aanwijst al 98,5%. Accuracy wordt gedomineerd door de grote groep gezonde patiënten.
2. **Het gaat om de positieve klasse.** Het doel is stroke-patiënten vinden. Precision en recall kijken alleen naar die groep; het enorme aantal true negatives telt niet mee.
3. **F1 dwingt een balans af.** Een model dat iedereen als risico aanwijst, heeft recall 1 maar precision 0,015, en dus een F1 van maar 0,03. Een model dat bijna niemand aanwijst, heeft een hoge precision maar recall ≈ 0. Omdat F1 een **harmonisch gemiddelde** is, wordt de score naar de laagste van de twee getrokken: alleen een model dat stroke-patiënten vindt **én** niet te veel vals alarm geeft, scoort hoog.
4. **Eén getal** maakt modellen en Kaggle-inzendingen eenvoudig vergelijkbaar.

**Kritische kanttekening:** F1 weegt een gemiste patiënt (FN) even zwaar als een vals alarm (FP). In de zorg is een gemiste patiënt vaak erger dan een extra controle. In de praktijk zou je daarom ook kunnen kiezen voor de F2-score, die recall zwaarder laat wegen. Voor deze competitie houden we F1 aan, maar we rapporteren steeds ook precision en recall.

Bron:(He & Garcia, 2009; Saito & Rehmsmeier, 2015)

## 5. Opdracht 3 – Modelleren en uitleg modellen

### 5.1 Imbalanced dataset en technieken
**Waarom is deze dataset imbalanced?** Een herseninfarct is in een willekeurige groep mensen een **zeldzame gebeurtenis**. De meeste mensen in de dataset zijn jong of middelbaar en hebben nooit een herseninfarct gehad. De data is dus een afspiegeling van de werkelijkheid: 98,5% zonder en 1,5% met stroke (ongeveer 1 op 64).

**Waarom is dat een probleem?** Een model minimaliseert de fouten over alle patiënten. Omdat 98,5% gezond is, is "altijd 0 voorspellen" al een bijna perfecte oplossing voor de loss-functie. De paar honderd stroke-patiënten hebben bijna geen invloed. Zonder maatregelen vindt het model daarom niemand (zie 4.3).

**Technieken.** Er zijn meerdere technieken om dit op te lossen (He & Garcia, 2009). Wij passen bij **ieder model minimaal twee** technieken toe:

| Techniek | Hoe werkt het? |
|---|---|
| **1. Class weights** (cost-sensitive learning) | Fouten op de kleine klasse tellen zwaarder in de loss. Met `class_weight="balanced"` krijgt klasse $k$ het gewicht $\frac{n}{2 \cdot n_k}$: een stroke-patiënt telt ongeveer 64 keer zo zwaar als een gezonde patiënt. Bij XGBoost heet dit `scale_pos_weight`. Er gaat geen data verloren. |
| **2. Random undersampling** | Uit de grote klasse wordt willekeurig een deel van de rijen weggelaten, tot een gekozen verhouding (bijv. 1 stroke op 4 gezonde patiënten). Het model ziet de kleine klasse dan relatief vaker. Voordeel: training wordt veel sneller. Nadeel: informatie van de weggelaten patiënten gaat verloren. |
| **3. Beslisdrempel afstemmen** (threshold moving) | Een model geeft een score; standaard voorspelt het "stroke" boven 0,5. Bij imbalance liggen bijna alle kansen ver onder 0,5. Door een andere drempel te kiezen (de drempel met de hoogste F1 op out-of-fold voorspellingen) verschuif je de balans tussen precision en recall. Het model zelf verandert niet (Lipton et al., 2014). |
| **Ondersteunend: stratificatie** | In iedere split en fold zit hetzelfde percentage stroke, zodat iedere fold genoeg positieve voorbeelden heeft. |

Een bekende alternatieve techniek is **oversampling**, bijvoorbeeld **SMOTE**: daarbij worden nieuwe, synthetische stroke-patiënten gemaakt door te interpoleren tussen een stroke-patiënt en zijn naaste stroke-buren (Chawla et al., 2002). In een eerste verkenning gaf dit bij deze data geen betere F1 dan undersampling en class weights, en het maakt de training trager. We gebruiken het daarom niet in de eindmodellen.

**Belangrijk:** samplers staan **in de pipeline** (`imblearn.pipeline.Pipeline`). Zo wordt alleen het trainingsdeel van iedere fold geresampled en blijven validatiefolds en validatieset onaangetast (Lemaître et al., 2017).

**Overzicht per model**

| Model | Techniek 1 | Techniek 2 | Techniek 3 |
|---|---|---|---|
| KNN | Random undersampling | Drempel afstemmen | – (KNN heeft geen class weights) |
| Logistic Regression | Class weights | Drempel afstemmen | – |
| SVM | Random undersampling | Class weights | Drempel afstemmen |
| Decision Tree | Class weights | Drempel afstemmen | – |
| Random Forest | Class weights (`balanced_subsample`) | Drempel afstemmen | – |
| Gradient Boosting | Random undersampling | Drempel afstemmen | – (geen class weights in scikit-learn) |
| XGBoost | Class weights (`scale_pos_weight`) | Drempel afstemmen | – |
| Eigen ensemble | Technieken van de losse modellen | Drempel afstemmen | – |

(Claude, 2026. Prompt 1: &lt;https://claude.ai/share/0f951b3c-8732-4003-9989-97107ceed719&gt;) , Hergeschreven

### 5.2 Werkwijze voor ieder model
Ieder model doorloopt dezelfde stappen via de klasse `ModelExperiment` (paragraaf 2.5):

1. **Cross-validation en hyperparameter tuning.** `GridSearchCV` probeert alle combinaties uit het *parameter grid*. Iedere combinatie wordt getest met **5-voudige gestratificeerde cross-validation**: de trainingsset wordt in 5 delen gesplitst; het model traint 5 keer op 4 delen en wordt getest op het vijfde deel. Het gemiddelde van de 5 scores bepaalt de beste combinatie. Zo hangt de keuze niet af van één toevallige split.
2. **Score voor het tunen.** We tunen op de **F1-score bij de beste drempel** (`F1_BEST_SCORER`). We stemmen de drempel daarna toch af; met de standaarddrempel 0,5 zouden we hyperparameters kiezen die toevallig goed bij die drempel passen. Deze CV-F1 is iets te optimistisch, omdat de drempel per fold wordt gekozen.
3. **Drempel kiezen.** Met `cross_val_predict` krijgt iedere trainingspatiënt een voorspelling van een model dat hem **niet** heeft gezien (*out-of-fold*). Op die scores kiezen we één drempel.
4. **Eerlijke test** op de validatieset met die vaste drempel → de **validatie-F1**. Dit is onze notebookscore.
5. **Kaggle:** het model met de beste hyperparameters en drempel wordt opnieuw getraind op alle data. De voorspellingen voor `test.csv` staan in de map `submissions/`.


In [ ]:
results = ResultsTable()
print(f"Folds: {N_FOLDS}, gestratificeerd | trainingsset: {len(y_train):,} "
      f"patiënten, waarvan {y_train.sum()} met stroke")
print(f"Verhouding gezond : stroke = "
      f"{(y_train == 0).sum() / y_train.sum():.1f} : 1")

### 5.4 Logistic Regression

#### 5.4.1 Hoe werkt logistic regression?
Ondanks de naam is logistic regression een **classificatiemodel**. Het model berekent eerst een gewogen som van de features, net als lineaire regressie:

$$z = b + w_1 x_1 + w_2 x_2 + \dots + w_p x_p = \mathbf{w}^\top \mathbf{x} + b$$

Daarna zet de **sigmoïdefunctie** $z$ om in een kans tussen 0 en 1:

$$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

$\hat{p}$ is de voorspelde kans op stroke. Ligt $\hat{p}$ boven de drempel (standaard 0,5, dus $z > 0$), dan voorspelt het model klasse 1. De beslisgrens $z = 0$ is een rechte lijn (een hypervlak): logistic regression is een **lineair model**. Een positief gewicht $w_j$ betekent dat een hogere waarde van feature $j$ de kans op stroke verhoogt. Daardoor is het model goed uit te leggen.

#### 5.4.2 Loss-functie: log-loss (binary cross-entropy)
De gewichten worden zo gekozen dat de **log-loss** zo klein mogelijk is:

$$J(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n} \Big[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \Big]$$

- Bij een patiënt **met** stroke ($y_i = 1$) blijft alleen $-\log(\hat{p}_i)$ over. De straf is klein als $\hat{p}_i$ dicht bij 1 ligt en loopt hard op als $\hat{p}_i$ naar 0 gaat.
- Bij een patiënt **zonder** stroke ($y_i = 0$) blijft $-\log(1 - \hat{p}_i)$ over. De straf is groot als het model een hoge kans voorspelt.
- Voorbeeld: $\hat{p} = 0{,}9$ bij $y = 1$ kost $-\ln(0{,}9) = 0{,}105$, maar $\hat{p} = 0{,}1$ bij $y = 1$ kost $-\ln(0{,}1) = 2{,}303$. Zeker zijn én fout zitten wordt dus zwaar bestraft.

Met `class_weight="balanced"` krijgt iedere patiënt een gewicht $s_i$ in de som: $J = -\frac{1}{n}\sum_i s_i\,[\dots]$. Fouten op stroke-patiënten tellen dan ongeveer 64 keer zo zwaar.

(Claude, 2026. Prompt 1: &lt; https://claude.ai/share/3f3afb79-0271-4ffc-97b6-6c669aeb2993 &gt;; James et al., 2021; Géron, 2022), Beetje hergeschreven. 

---

#### 5.4.3 Gradient descent
Voor log-loss bestaat geen formule die de beste gewichten in één keer geeft. Daarom zoekt het algoritme ze **stap voor stap** met **gradient descent**:

1. Begin met startwaarden voor de gewichten (bijvoorbeeld allemaal 0).
2. Bereken de **gradiënt**: de afgeleide van de loss naar ieder gewicht. Die wijst in de richting waarin de loss het snelst **stijgt**.
3. Zet een kleine stap in de **tegenovergestelde richting**. De stapgrootte is de **learning rate** $\alpha$.
4. Herhaal tot de loss nauwelijks nog daalt (*convergentie*).

Voor log-loss zijn de gradiënten:

$$\frac{\partial J}{\partial w_j} = \frac{1}{n}\sum_{i=1}^{n} (\hat{p}_i - y_i)\, x_{ij} \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_{i=1}^{n} (\hat{p}_i - y_i)$$

En de update-regels:

$$w_j \leftarrow w_j - \alpha \frac{\partial J}{\partial w_j} \qquad\qquad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

Een te grote $\alpha$ laat het algoritme over het minimum heen springen; een te kleine $\alpha$ maakt het heel traag.

**Eenvoudig rekenvoorbeeld.** Vier patiënten met één feature, de gestandaardiseerde leeftijd: $x = [-1{,}5;\ -0{,}5;\ 0{,}5;\ 1{,}5]$ met $y = [0;\ 0;\ 1;\ 1]$ (de twee oudste patiënten hebben stroke). We starten met $w = 0$, $b = 0$ en $\alpha = 0{,}5$.

*Iteratie 1*
- $z_i = 0 \cdot x_i + 0 = 0$, dus $\hat{p}_i = \sigma(0) = 0{,}5$ voor alle vier.
- Loss: $J = -\frac{1}{4}\cdot 4 \cdot \ln(0{,}5) = 0{,}693$.
- Fouten: $\hat{p}_i - y_i = [0{,}5;\ 0{,}5;\ -0{,}5;\ -0{,}5]$.
- $\frac{\partial J}{\partial w} = \frac{1}{4}\big(0{,}5 \cdot (-1{,}5) + 0{,}5 \cdot (-0{,}5) + (-0{,}5) \cdot 0{,}5 + (-0{,}5) \cdot 1{,}5\big) = \frac{-2}{4} = -0{,}5$
- $\frac{\partial J}{\partial b} = \frac{1}{4}(0{,}5 + 0{,}5 - 0{,}5 - 0{,}5) = 0$
- Update: $w = 0 - 0{,}5 \cdot (-0{,}5) = 0{,}25$ en $b = 0 - 0{,}5 \cdot 0 = 0$.

*Iteratie 2*
- $z = 0{,}25 \cdot x = [-0{,}375;\ -0{,}125;\ 0{,}125;\ 0{,}375]$, dus $\hat{p} = [0{,}407;\ 0{,}469;\ 0{,}531;\ 0{,}593]$.
- De oudere patiënten krijgen nu al een hogere kans en de loss daalt naar 0,578.

De code hieronder voert dezelfde stappen uit met zelfgeschreven functies en laat zien dat de loss blijft dalen en $w$ blijft groeien.

(Claude, 2026. Prompt 2: &lt; https://claude.ai/share/3f3afb79-0271-4ffc-97b6-6c669aeb2993 &gt;; Géron, 2022)

In [ ]:
# Prompt 2: < https://claude.ai/share/3f3afb79-0271-4ffc-97b6-6c669aeb2993 >
def sigmoid(z):
    """Sigmoïdefunctie: zet een getal om naar een kans tussen 0 en 1."""
    return 1 / (1 + np.exp(-z))


def log_loss_manual(y_true, p_pred):
    """Log-loss (binary cross-entropy) volgens de formule in 5.4.2."""
    return -np.mean(y_true * np.log(p_pred)
                    + (1 - y_true) * np.log(1 - p_pred))


def gradient_descent(x, y, learning_rate=0.5, n_iterations=6):
    """Gradient descent voor logistic regression met één feature."""
    weight, bias = 0.0, 0.0
    history = []
    for iteration in range(1, n_iterations + 1):
        p_pred = sigmoid(weight * x + bias)
        gradient_w = np.mean((p_pred - y) * x)
        gradient_b = np.mean(p_pred - y)
        history.append({"iteratie": iteration, "w": weight, "b": bias,
                        "loss": log_loss_manual(y, p_pred),
                        "dJ/dw": gradient_w, "dJ/db": gradient_b})
        # Stap tegen de gradiënt in
        weight -= learning_rate * gradient_w
        bias -= learning_rate * gradient_b
    return pd.DataFrame(history)


x_demo = np.array([-1.5, -0.5, 0.5, 1.5])
y_demo = np.array([0, 0, 1, 1])
gradient_descent(x_demo, y_demo).round(4)

----

#### 5.4.4 Regularisatie bij logistic regression

**Wat is regularisatie en waarom gebruik je het?**
Logistic regression leert voor elke feature een gewicht $w_j$. Als het model te veel vrijheid heeft, kunnen die gewichten erg groot worden. Het model past zich dan te sterk aan de trainingsdata aan, inclusief ruis (**overfitting**), en werkt slechter op nieuwe data.

Regularisatie voegt een **strafterm** toe aan de verliesfunctie. Het model krijgt zo een straf voor grote gewichten en kiest daardoor voor een eenvoudiger model dat beter generaliseert (Hastie et al., 2009).

De algemene vorm is:

$$
J(w) = \underbrace{-\frac{1}{n}\sum_{i=1}^{n} \big[ y_i \log(p_i) + (1-y_i)\log(1-p_i) \big]}_{\text{log-loss}} + \lambda \cdot \text{strafterm}(w)
$$

Hierin is $p_i$ de voorspelde kans op klasse 1 en bepaalt $\lambda$ hoe zwaar de straf meetelt.

**L2, L1 en Elastic Net**

| Type | Strafterm | Wat doet het? |
|---|---|---|
| **L2 (Ridge)** | $\lambda \sum_j w_j^2$ | Maakt alle gewichten kleiner, maar bijna nooit precies 0. Alle features blijven meedoen. Werkt goed als features met elkaar samenhangen. |
| **L1 (Lasso)** | $\lambda \sum_j \lvert w_j \rvert$ | Kan gewichten **precies 0** maken. Het model kiest daardoor zelf welke features het gebruikt (feature selectie). |
| **Elastic Net** | $\lambda \big[ \rho \sum_j \lvert w_j \rvert + \tfrac{1-\rho}{2} \sum_j w_j^2 \big]$ | Een mix van L1 en L2. Kan gewichten op 0 zetten, maar blijft stabieler dan L1 als features sterk samenhangen. |

Kort gezegd: L2 **verkleint**, L1 **verkleint en selecteert**, Elastic Net doet **allebei** (Hastie et al., 2009).

**C en l1_ratio in scikit-learn**
In scikit-learn wordt de sterkte van de regularisatie niet met $\lambda$ ingesteld, maar met **C**. C is het omgekeerde van $\lambda$ ($C \approx 1/\lambda$):

- **Kleine C** (bijv. 0,01) → **sterke** regularisatie → kleinere gewichten, eenvoudiger model.
- **Grote C** (bijv. 100) → **zwakke** regularisatie → het model volgt de trainingsdata meer.

**l1_ratio** (de $\rho$ in de formule hierboven) bepaalt bij Elastic Net de verhouding tussen L1 en L2:

- `l1_ratio = 0` → alleen L2
- `l1_ratio = 1` → alleen L1
- daartussen (bijv. 0,5) → een mix

Elastic Net werkt in scikit-learn met de solver `saga` (scikit-learn developers, z.d.).

**Let op:** de strafterm hangt af van de grootte van de gewichten, en die hangt weer af van de schaal van de features. Schaal de features daarom eerst (bijv. met `StandardScaler`), zodat elke feature eerlijk wordt bestraft. C en l1_ratio zoeken we met cross-validation.

(Claude, 2026. Prompt 2: <https://claude.ai/share/0f951b3c-8732-4003-9989-97107ceed719>; Hastie et al., 2009; scikit-learn developers, z.d.)

#### 5.4.5 Hoe verhoudt regularisatie zich tot overfitting?
**Overfitting** betekent dat een model de ruis in de trainingsdata leert in plaats van het echte patroon: goed op de trainingsdata, slecht op nieuwe data. Grote gewichten maken een model gevoelig voor kleine veranderingen in de input (**hoge variantie**). Regularisatie remt de gewichten af en **verlaagt de variantie**, in ruil voor iets meer **bias**: de *bias-variance trade-off*.

- **Te weinig regularisatie** (grote C): kans op overfitting, zeker met veel features en weinig voorbeelden van de kleine klasse.
- **Te veel regularisatie** (zeer kleine C): alle gewichten gaan naar 0 en het model leert te weinig → **underfitting**.

Met cross-validation zoeken we de C die daartussen het beste werkt.

#### 5.4.6 Hyperparameter tuning
**Pipeline:** `StandardScaler` → `LogisticRegression` (solver `saga`, die L1, L2 en Elastic Net ondersteunt).

**Technieken tegen imbalance:** (1) `class_weight="balanced"`; (2) beslisdrempel afstemmen.

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `C` | 0,001 / 0,01 / 0,1 / 1 / 10 | Inverse regularisatiesterkte |
| `l1_ratio` | 0 / 0,5 / 1 | 0 = L2, 0,5 = Elastic Net, 1 = L1 |

In [ ]:
# penalty="elasticnet" + l1_ratio werkt in alle recente versies van
# scikit-learn: l1_ratio 0 is L2, 1 is L1 en daartussen Elastic Net.
log_reg = ModelExperiment(
    name="Logistic Regression",
    pipeline=Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="elasticnet", solver="saga", class_weight="balanced",
            max_iter=5000, random_state=RANDOM_STATE))]),
    param_grid={"model__C": [0.001, 0.01, 0.1, 1, 10],
                "model__l1_ratio": [0, 0.5, 1]},
    techniques="class weights + drempel").run(data)
results.add(log_reg)

In [ ]:
log_reg.plot_evaluation(y_val)

# Gewichten van het beste model: welke features verhogen het risico?
coefficients = pd.Series(
    log_reg.best_model.named_steps["model"].coef_[0],
    index=X_train.columns).sort_values()
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(coefficients.index, coefficients.values,
        color=np.where(coefficients > 0, "#C44E52", "#4C72B0"))
ax.set(title="Gewichten logistic regression (gestandaardiseerde features)",
       xlabel="Gewicht (rood = verhoogt risico, blauw = verlaagt risico)")
plt.tight_layout()
plt.show()

**Resultaat logistic regression.** De beste combinatie is **C = 0,01** met **l1_ratio = 0**, dus vrij sterke **L2-regularisatie**.

Op de validatieset haalt logistic regression een F1 van **0,131** (precision 0,079; recall 0,379). Het model vindt 39 van de 103 stroke-patiënten, met 455 vals alarmen. Drempelafstemming verhoogt de F1 van 0,082 naar 0,131. De drempel is 0,80, omdat de class weights alle kansen sterk omhoog duwen.

De gewichten kloppen met de EDA: **leeftijd** heeft verreweg het grootste gewicht, gevolgd door glucose, hoge bloeddruk, roken en hartziekte; `work_type_children` verlaagt het risico. `ever_married_Yes` heeft nu een klein **negatief** gewicht. Het positieve verband uit de EDA kwam dus door leeftijd, en daarvoor corrigeert het model.

---

### 5.5 Support Vector Machine (SVM)
#### 5.5.1 Hoe werkt een SVM?

Een Support Vector Machine (SVM) zoekt een grens die de twee klassen zo goed mogelijk scheidt. Bij een SVM gebruiken we de labels **$y = -1$** (geen stroke) en **$y = +1$** (stroke).

**Het hypervlak**
De grens is een **hypervlak**: bij 2 features een lijn, bij 3 features een vlak, enzovoort. Het hypervlak bestaat uit alle punten $x$ waarvoor geldt:

$$
f(x) = w \cdot x + b = 0
$$

- $w$ is de vector met gewichten (de richting van de grens),
- $b$ is de bias (verschuiving).

Een nieuwe patiënt krijgt label $+1$ als $f(x) > 0$ en label $-1$ als $f(x) < 0$ (James et al., 2021).

**De marge**
Er zijn veel lijnen die de klassen scheiden. De SVM kiest de lijn met de **grootste marge**: de grootste "lege strook" tussen de klassen. De randen van die strook liggen bij $f(x) = +1$ en $f(x) = -1$. De breedte van de marge is:

$$
\text{marge} = \frac{2}{\lVert w \rVert}
$$

Een grote marge betekent dus een kleine $\lVert w \rVert$. Daarom minimaliseert de SVM $\tfrac{1}{2}\lVert w \rVert^2$, met als eis dat elk punt aan de goede kant ligt en buiten de marge:

$$
y_i \,(w \cdot x_i + b) \geq 1 \quad \text{voor alle } i
$$

**Support vectors**
De **support vectors** zijn de punten die precies op de rand van de marge liggen, of erin of aan de verkeerde kant. Alleen deze punten bepalen waar het hypervlak ligt. Als je een punt ver van de grens weghaalt, verandert er niets (Cortes & Vapnik, 1995).

**Soft margin: C en slack**
In echte data (zoals onze stroke-data) overlappen de klassen. Een perfecte scheiding bestaat dan niet. Daarom staat de **soft margin** fouten toe via **slackvariabelen** $\xi_i \geq 0$ (Cortes & Vapnik, 1995):

$$
\min_{w,\,b,\,\xi} \; \frac{1}{2}\lVert w \rVert^2 + C \sum_{i=1}^{n} \xi_i
\quad \text{met} \quad y_i\,(w \cdot x_i + b) \geq 1 - \xi_i
$$

De slack is gelijk aan de **hinge loss**: $\xi_i = \max\big(0,\; 1 - y_i f(x_i)\big)$.

| $y_i f(x_i)$ | $\xi_i$ | Betekenis |
|---|---|---|
| $\geq 1$ | $0$ | Goed geclassificeerd, buiten de marge |
| tussen 0 en 1 | tussen 0 en 1 | Goed geclassificeerd, maar binnen de marge |
| $< 0$ | $> 1$ | Fout geclassificeerd |

**C** bepaalt hoe zwaar fouten meetellen:
- **Grote C** → fouten zijn duur → smalle marge, model volgt de trainingsdata sterk (risico op overfitting).
- **Kleine C** → fouten zijn goedkoop → brede marge, eenvoudiger model (James et al., 2021).

Omdat de marge afhangt van afstanden, moeten de features eerst **gestandaardiseerd** worden.

(Claude, 2026. Prompt 3: <https://claude.ai/share/0f951b3c-8732-4003-9989-97107ceed719>; Cortes & Vapnik, 1995; James et al., 2021)

In [ ]:
# Claude, 2026. Prompt 3: https://claude.ai/share/0f951b3c-8732-4003-9989-97107ceed719
# Klein rekenvoorbeeld: lineaire SVM op 20 patiënten met 2 features

features = ["age", "avg_glucose_level"]

# 10 patiënten met en 10 zonder stroke uit de trainingsdata
df = X_train[features].copy()
df["stroke"] = np.asarray(y_train)

met_stroke = df[df["stroke"] == 1].sample(10, random_state=RANDOM_STATE)
zonder_stroke = df[df["stroke"] == 0].sample(10, random_state=RANDOM_STATE)
mini = pd.concat([met_stroke, zonder_stroke])

# Standaardiseren en labels omzetten naar -1 / +1
X_mini = StandardScaler().fit_transform(mini[features])
y_mini = np.where(mini["stroke"] == 1, 1, -1)

# Lineaire SVM trainen
svm_demo = SVC(kernel="linear", C=1, random_state=RANDOM_STATE)
svm_demo.fit(X_mini, y_mini)

w_demo = svm_demo.coef_[0]
b_demo = svm_demo.intercept_[0]
norm_w = np.linalg.norm(w_demo)

print(f"w            = {np.round(w_demo, 3)}")
print(f"b            = {b_demo:.3f}")
print(f"||w||        = {norm_w:.3f}")
print(f"marge 2/||w||= {2 / norm_w:.3f}")
print(f"aantal support vectors = {len(svm_demo.support_)} "
      f"(geen stroke: {svm_demo.n_support_[0]}, stroke: {svm_demo.n_support_[1]})")

# Tabel per patiënt: f(x), y·f(x) en hinge loss
f_x = svm_demo.decision_function(X_mini)          # f(x) = w·x + b
y_f = y_mini * f_x
hinge = np.maximum(0, 1 - y_f)

tabel = pd.DataFrame({
    "age (std)": X_mini[:, 0],
    "glucose (std)": X_mini[:, 1],
    "y": y_mini,
    "f(x)": f_x,
    "y·f(x)": y_f,
    "hinge loss": hinge,
    "support vector": np.isin(np.arange(len(y_mini)), svm_demo.support_),
}).round(3)

display(tabel)
print(f"Totale hinge loss (som van de slack): {hinge.sum():.3f}")

# Zelf narekenen voor één patiënt: f(x) = w·x + b
i = 19
eigen_f = w_demo[0] * X_mini[i, 0] + w_demo[1] * X_mini[i, 1] + b_demo
print(f"Patiënt {i}: f(x) = {w_demo[0]:.3f} * {X_mini[i, 0]:.3f} + "
      f"{w_demo[1]:.3f} * {X_mini[i, 1]:.3f} + ({b_demo:.3f}) = {eigen_f:.3f} "
      f"(scikit-learn: {f_x[i]:.3f})")


**Interpretatie rekenvoorbeeld.** De SVM vindt $\mathbf{w} = (1{,}696;\ 0{,}035)$ en $b = -0{,}263$. Het gewicht van leeftijd is veel groter dan dat van glucose: in deze kleine steekproef scheidt vooral leeftijd de twee klassen. De marge is $2 / \lVert\mathbf{w}\rVert = 2 / 1{,}696 = 1{,}179$ (in gestandaardiseerde eenheden).

De tabel laat de situaties uit paragraaf 5.5.3 zien:
- **Patiënt 0** ligt goed en buiten de marge: $y \cdot f(\mathbf{x}) = 1{,}577 \ge 1$, dus hinge loss 0. Dit punt heeft geen invloed op het hypervlak.
- **Patiënten 1 en 14** liggen precies op de marge ($y \cdot f(\mathbf{x}) = 1{,}000$): het zijn support vectors met loss 0.
- **Patiënten 4, 6 en 12** liggen aan de goede kant, maar binnen de marge ($0 < y \cdot f(\mathbf{x}) < 1$): een loss tussen 0 en 1.
- **Patiënt 19** heeft geen stroke ($y = -1$), maar $f(\mathbf{x}) = 0{,}230 > 0$: hij is fout geclassificeerd en de hinge loss is $1 - (-1)(0{,}230) = 1{,}230$.

Alleen de 6 support vectors (op, binnen of aan de verkeerde kant van de marge) bepalen waar het hypervlak ligt.

#### 5.5.2 Kernel en kernel trick

**Waarom een kernel?**
Een lineaire SVM zoekt een rechte grens (hypervlak). Bij veel datasets, zoals onze stroke-data, zijn de klassen niet met een rechte lijn te scheiden. Een oplossing is de data eerst naar een **hogere dimensie** te brengen met een transformatie $\varphi(x)$. In die nieuwe ruimte is een rechte scheiding vaak wél mogelijk. Terug in de originele ruimte is die grens dan gebogen (Hastie et al., 2009).

**Wat is een kernel?**
Het probleem: $\varphi(x)$ uitrekenen kan heel duur zijn, omdat de nieuwe ruimte heel veel (soms oneindig veel) dimensies heeft. Maar de SVM heeft $\varphi(x)$ zelf niet nodig. In de duale formulering gebruikt hij alleen **inproducten** tussen punten. Een **kernel** is een functie die dat inproduct in de nieuwe ruimte direct uitrekent vanuit de originele punten:

$$
K(x, z) = \varphi(x) \cdot \varphi(z)
$$

**Wat doet de kernel trick?**
De **kernel trick** is: overal waar de SVM een inproduct $x \cdot z$ gebruikt, vervangen we dat door $K(x, z)$. De beslisfunctie wordt dan:

$$
f(x) = \sum_{i \in S} \alpha_i \, y_i \, K(x_i, x) + b
$$

met $S$ de support vectors en $\alpha_i$ hun gewichten. Zo werkt het model alsof de data in de hogere dimensie staat, zonder dat we $\varphi(x)$ ooit uitrekenen. Dat is veel sneller (Géron, 2022).

**Veelgebruikte kernels**

| Kernel | Formule | In scikit-learn | Kenmerk |
|---|---|---|---|
| **Lineair** | $K(x,z) = x \cdot z$ | `kernel="linear"` | Geen transformatie; rechte grens. |
| **Polynomiaal** | $K(x,z) = (\gamma \, x \cdot z + r)^d$ | `kernel="poly"` (`degree`, `gamma`, `coef0`) | Gebogen grens; hogere $d$ = complexere grens. |
| **RBF (Gaussisch)** | $K(x,z) = \exp\!\big(-\gamma \lVert x - z \rVert^2\big)$ | `kernel="rbf"` (`gamma`) | Meet hoe dicht punten bij elkaar liggen; zeer flexibel. Grote $\gamma$ = grillige grens (risico op overfitting), kleine $\gamma$ = gladde grens. |

De RBF-kernel hoort bij een ruimte met oneindig veel dimensies. Daar is $\varphi(x)$ dus niet uit te rekenen, maar met de kernel trick werkt het toch (Géron, 2022; Hastie et al., 2009).

**Rekenvoorbeeld**
Neem $x = (1, 2)$, $z = (3, 1)$ en de kernel $K(x, z) = (x \cdot z)^2$.

**Manier 1: met de kernel (direct in 2D)**

$$
x \cdot z = 1 \cdot 3 + 2 \cdot 1 = 5 \qquad\Rightarrow\qquad K(x, z) = 5^2 = \mathbf{25}
$$

**Manier 2: eerst transformeren met $\varphi$ (naar 3D)**

Bij deze kernel hoort de transformatie:

$$
\varphi(x) = \big(x_1^2,\; \sqrt{2}\,x_1 x_2,\; x_2^2\big)
$$

$$
\varphi(x) = \big(1^2,\; \sqrt{2}\cdot 1 \cdot 2,\; 2^2\big) = \big(1,\; 2\sqrt{2},\; 4\big)
$$

$$
\varphi(z) = \big(3^2,\; \sqrt{2}\cdot 3 \cdot 1,\; 1^2\big) = \big(9,\; 3\sqrt{2},\; 1\big)
$$

$$
\varphi(x) \cdot \varphi(z) = 1 \cdot 9 + 2\sqrt{2} \cdot 3\sqrt{2} + 4 \cdot 1 = 9 + 12 + 4 = \mathbf{25}
$$

Beide manieren geven **25**. De kernel kwam er met één inproduct en één kwadraat, zonder de 3D-vectoren te maken. Bij veel features en hogere graden scheelt dat enorm veel rekenwerk.

(Claude, 2026. Prompt 4: <https://claude.ai/share/0f951b3c-8732-4003-9989-97107ceed719>; Hastie et al., 2009; Géron, 2022)

#### 5.5.3 Loss-functie: hinge loss
Een SVM leert door een **hinge loss** te minimaliseren, samen met een **L2-strafterm** op de gewichten. Met labels $y_i \in \{-1, +1\}$ en $f(x) = w \cdot x + b$ is het doel (Cortes & Vapnik, 1995; scikit-learn developers, z.d.):

$$
\min_{w,\,b} \; \underbrace{\frac{1}{2}\lVert w \rVert^2}_{\text{L2-strafterm}} \;+\; C \sum_{i=1}^{n} \underbrace{\max\big(0,\; 1 - y_i f(x_i)\big)}_{\text{hinge loss}}
$$

- De **strafterm** $\tfrac{1}{2}\lVert w \rVert^2$ houdt de gewichten klein, en dus de marge ($2/\lVert w \rVert$) groot.
- De **hinge loss** straft punten die fout liggen of te dicht bij de grens.
- **C** bepaalt hoe zwaar de fouten meetellen ten opzichte van de strafterm.

Wat de hinge loss per situatie betekent, staat in de tabel in 5.5.1 en in het rekenvoorbeeld.

**Verschil met log-loss**
Logistic regression gebruikt de **log-loss**. Met labels $-1$ en $+1$ is die:

$$
\text{log-loss} = \log\big(1 + e^{-y \cdot f(x)}\big)
$$

| | Hinge loss (SVM) | Log-loss (logistic regression) |
|---|---|---|
| Straf bij $y \cdot f(x) \geq 1$ | Precies $0$ | Klein, maar nooit $0$ |
| Welke punten bepalen het model? | Alleen de support vectors | Alle punten |
| Uitkomst | Een score $f(x)$, geen kans | Een kans tussen 0 en 1 |

Een SVM geeft dus standaard geen kansen. In scikit-learn kan dat wel met `probability=True`, maar die kansen worden achteraf met extra cross-validation geschat (scikit-learn developers, z.d.). Daarom gebruiken wij bij de SVM de score $f(x)$ (`decision_function`) om de drempel te kiezen.

